In [1]:
import os
os.chdir("..")

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from src.shared.models import Database
from src.shared.logger import Logger
from src.shared.constants import *
from datetime import datetime

# Timestamp for filenames
CURRENT_TIME = datetime.now().strftime("%d%m%y_%H%M%S")

# ------------------------------
# Logger setup
# ------------------------------
log_file_path = LOG_DIR / f"shopify_fetch_{CURRENT_TIME}.log"  # not used in Logger
logger_instance = Logger(name=f"shopify_fetch_{CURRENT_TIME}", log_folder=LOG_DIR)

db = Database(
        host=os.getenv("HOST"),
        port=os.getenv("PORT"),
        database=os.getenv("DATABASE"),
        user=os.getenv("USER"),
        password=os.getenv("PASSWORD"),
        logger=logger_instance
    )

query = f"""
SELECT 
    UPPER(s.stk_pattern) as stk_pattern,
    FLOOR(s.stk_weight) AS weight_rounded,
    UPPER(s.stk_type) || ' | ' || UPPER(s.stk_pattern) || ' | ' || FLOOR(s.stk_weight) AS map_key,
    MAX(s.stk_labor_cost) AS max_labor_cost
FROM konghin.stock s
JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id
WHERE p.pur_date >= NOW() - INTERVAL '365 days'
GROUP BY
    map_key,
    stk_pattern,
    FLOOR(s.stk_weight)
ORDER BY 
    stk_pattern,
    weight_rounded;
"""


recent_labor = db.select_raw(query)
recent_labor

[2025-12-11 17:00:34] [DEBUG] Logger initialized at D:\development\JQ\PriceUpdateAlgo\algo_docker\logs\shopify_fetch_111225_170034.log (UTF-8 safe)
[2025-12-11 17:00:34] [INFO] Successfully executed query: 
SELECT 
    UPPER(s.stk_pattern) as stk_pattern,
    FLOOR(s.stk_weight) AS weight_rounded,
    UPPER(s.stk_type) || ' | ' || UPPER(s.stk_pattern) || ' | ' || FLOOR(s.stk_weight) AS map_key,
    MAX(s.stk_labor_cost) AS max_labor_cost
FROM konghin.stock s
JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id
WHERE p.pur_date >= NOW() - INTERVAL '365 days'
GROUP BY
    map_key,
    stk_pattern,
    FLOOR(s.stk_weight)
ORDER BY 
    stk_pattern,
    weight_rounded;



,stk_pattern,weight_rounded,map_key,max_labor_cost
0,3D平安锁,3,CHARM | 3D平安锁 | 3,50
1,3D平安锁,4,CHARM | 3D平安锁 | 4,50
2,3D平安锁1,3,CHARM | 3D平安锁1 | 3,45
3,ANGGUR,19,NECKLACE | ANGGUR | 19,234
4,ANGGUR间圆圈,5,BRACELET | ANGGUR间圆圈 | 5,85
...,...,...,...,...
173,龙链,7,BRACELET | 龙链 | 7,50.0
174,龙链,8,BRACELET | 龙链 | 8,138
175,龙链,9,BRACELET | 龙链 | 9,65.0
176,龙链,14,BRACELET | 龙链 | 14,75.0


In [4]:
recent_labor.loc[recent_labor['stk_pattern']=='龙链']

,stk_pattern,weight_rounded,map_key,max_labor_cost
168,龙链,2,BRACELET | 龙链 | 2,30.0
169,龙链,3,BRACELET | 龙链 | 3,35.0
170,龙链,4,BRACELET | 龙链 | 4,40.0
171,龙链,5,BRACELET | 龙链 | 5,40.0
172,龙链,6,BRACELET | 龙链 | 6,45.0
173,龙链,7,BRACELET | 龙链 | 7,50.0
174,龙链,8,BRACELET | 龙链 | 8,138
175,龙链,9,BRACELET | 龙链 | 9,65.0
176,龙链,14,BRACELET | 龙链 | 14,75.0


In [5]:
all_stk_query = """
SELECT
    s.stk_id,
    UPPER(s.stk_pattern) as stk_pattern,
    s.stk_weight,
    UPPER(s.stk_type) || ' | ' || UPPER(s.stk_pattern) || ' | ' || FLOOR(s.stk_weight) AS map_key,
    s.stk_labor_cost,
    p.pur_id,
    p.pur_date
FROM
    konghin.stock s
    JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id
WHERE UPPER(s.stk_status) = 'IN STOCK'
"""
all_stk = db.select_raw(all_stk_query)
all_stk

[2025-12-11 17:00:35] [INFO] Successfully executed query: 
SELECT
    s.stk_id,
    UPPER(s.stk_pattern) as stk_pattern,
    s.stk_weight,
    UPPER(s.stk_type) || ' | ' || UPPER(s.stk_pattern) || ' | ' || FLOOR(s.stk_weight) AS map_key,
    s.stk_labor_cost,
    p.pur_id,
    p.pur_date
FROM
    konghin.stock s
    JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id
WHERE UPPER(s.stk_status) = 'IN STOCK'



,stk_id,stk_pattern,stk_weight,map_key,stk_labor_cost,pur_id,pur_date
0,STK_100229,空心万字手链,6.06,BRACELET | 空心万字手链 | 6,79,PUR_100605,2024-10-01
1,STK_100154,啤片单扣手链,10.88,BRACELET | 啤片单扣手链 | 10,98,PUR_100959,2023-07-01
2,STK_100185,实心单扣手链,23.08,BRACELET | 实心单扣手链 | 23,125,PUR_100939,2024-12-01
3,STK_100186,空心双扣手链,2.83,BRACELET | 空心双扣手链 | 2,58,PUR_100939,2024-12-01
4,STK_100188,空心双扣手链,2.83,BRACELET | 空心双扣手链 | 2,89,PUR_100612,2024-10-01
...,...,...,...,...,...,...,...
709,STK_100927,SNOW FLAKE,1.69,PENDANT | SNOW FLAKE | 1,30,PUR_101008,2024-11-01
710,STK_100918,镂空圣诞袜,1.23,CHARM | 镂空圣诞袜 | 1,18,PUR_101036,2022-12-01
711,STK_100928,立体耶稣十字架2,4.45,PENDANT | 立体耶稣十字架2 | 4,9,PUR_100391,2014-07-01
712,STK_100922,ITALIC,6.61,BANGLE | ITALIC | 6,118,PUR_101034,2025-07-01


In [6]:
import pandas as pd
merged_full = pd.merge(all_stk,recent_labor[['map_key','max_labor_cost']],how='left',on=['map_key'])
merged_full

,stk_id,stk_pattern,stk_weight,map_key,stk_labor_cost,pur_id,pur_date,max_labor_cost
0,STK_100229,空心万字手链,6.06,BRACELET | 空心万字手链 | 6,79,PUR_100605,2024-10-01,NaN
1,STK_100154,啤片单扣手链,10.88,BRACELET | 啤片单扣手链 | 10,98,PUR_100959,2023-07-01,NaN
2,STK_100185,实心单扣手链,23.08,BRACELET | 实心单扣手链 | 23,125,PUR_100939,2024-12-01,NaN
3,STK_100186,空心双扣手链,2.83,BRACELET | 空心双扣手链 | 2,58,PUR_100939,2024-12-01,NaN
4,STK_100188,空心双扣手链,2.83,BRACELET | 空心双扣手链 | 2,89,PUR_100612,2024-10-01,NaN
...,...,...,...,...,...,...,...,...
709,STK_100927,SNOW FLAKE,1.69,PENDANT | SNOW FLAKE | 1,30,PUR_101008,2024-11-01,NaN
710,STK_100918,镂空圣诞袜,1.23,CHARM | 镂空圣诞袜 | 1,18,PUR_101036,2022-12-01,NaN
711,STK_100928,立体耶稣十字架2,4.45,PENDANT | 立体耶稣十字架2 | 4,9,PUR_100391,2014-07-01,NaN
712,STK_100922,ITALIC,6.61,BANGLE | ITALIC | 6,118,PUR_101034,2025-07-01,118


In [7]:
merged_found = merged_full.loc[merged_full['max_labor_cost'].notnull()]
merged_not_found = merged_full.loc[merged_full['max_labor_cost'].isnull()]

In [8]:
merged_not_found

,stk_id,stk_pattern,stk_weight,map_key,stk_labor_cost,pur_id,pur_date,max_labor_cost
0,STK_100229,空心万字手链,6.06,BRACELET | 空心万字手链 | 6,79,PUR_100605,2024-10-01,NaN
1,STK_100154,啤片单扣手链,10.88,BRACELET | 啤片单扣手链 | 10,98,PUR_100959,2023-07-01,NaN
2,STK_100185,实心单扣手链,23.08,BRACELET | 实心单扣手链 | 23,125,PUR_100939,2024-12-01,NaN
3,STK_100186,空心双扣手链,2.83,BRACELET | 空心双扣手链 | 2,58,PUR_100939,2024-12-01,NaN
4,STK_100188,空心双扣手链,2.83,BRACELET | 空心双扣手链 | 2,89,PUR_100612,2024-10-01,NaN
...,...,...,...,...,...,...,...,...
700,STK_100917,CANDY CANE,1.19,CHARM | CANDY CANE | 1,18,PUR_100188,2022-03-01,NaN
703,STK_100920,十字架,0.77,PENDANT | 十字架 | 0,10,PUR_100482,2024-08-01,NaN
709,STK_100927,SNOW FLAKE,1.69,PENDANT | SNOW FLAKE | 1,30,PUR_101008,2024-11-01,NaN
710,STK_100918,镂空圣诞袜,1.23,CHARM | 镂空圣诞袜 | 1,18,PUR_101036,2022-12-01,NaN


In [9]:
## Find Min Max of each stk_pattern (latest year)
percentage = 0.2

recent_min_max_query = f"""
SELECT
    UPPER(s.stk_pattern) as stk_pattern,
    MIN(FLOOR(s.stk_weight)) as min_weight,
    MAX(CEIL(s.stk_weight)) as max_weight,
    CEIL((MAX(CEIL(s.stk_weight)) - MIN(FLOOR(s.stk_weight))) * {percentage}) as range,
    UPPER(s.stk_type) || ' | ' || UPPER(s.stk_pattern) || ' | ' || FLOOR(s.stk_weight) AS map_key
FROM
    konghin.stock s
    JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id
WHERE p.pur_date >= NOW() - INTERVAL '365 days'
GROUP BY 
    map_key,
    UPPER(s.stk_pattern)
"""

recent_min_max = db.select_raw(recent_min_max_query)
recent_min_max

[2025-12-11 17:00:35] [INFO] Successfully executed query: 
SELECT
    UPPER(s.stk_pattern) as stk_pattern,
    MIN(FLOOR(s.stk_weight)) as min_weight,
    MAX(CEIL(s.stk_weight)) as max_weight,
    CEIL((MAX(CEIL(s.stk_weight)) - MIN(FLOOR(s.stk_weight))) * 0.2) as range,
    UPPER(s.stk_type) || ' | ' || UPPER(s.stk_pattern) || ' | ' || FLOOR(s.stk_weight) AS map_key
FROM
    konghin.stock s
    JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id
WHERE p.pur_date >= NOW() - INTERVAL '365 days'
GROUP BY 
    map_key,
    UPPER(s.stk_pattern)



,stk_pattern,min_weight,max_weight,range,map_key
0,圆边礼物盒,1,2,1,PENDANT | 圆边礼物盒 | 1
1,COCO,16,17,1,BRACELET | COCO | 16
2,人字链,3,4,1,BRACELET | 人字链 | 3
3,双层福字,1,2,1,CHARM | 双层福字 | 1
4,PAPER CLIP,5,6,1,BRACELET | PAPER CLIP | 5
...,...,...,...,...,...
173,NABILA,7,8,1,RING | NABILA | 7
174,实心单扣手链,15,16,1,BRACELET | 实心单扣手链 | 15
175,乾坤圈,4,5,1,PENDANT | 乾坤圈 | 4
176,CARTIER CHOCKER,5,6,1,NECLACE | CARTIER CHOCKER | 5


In [13]:
## find the optimum labor cost
df = pd.merge(merged_not_found,recent_min_max[['stk_pattern','range']],how='left',on='stk_pattern')
sample = df.iloc[:1]
if pd.isnull(sample['max_labor_cost'].iloc[0]):
    display(sample.iloc[:1])
    sample_stk_pattern = sample['stk_pattern'].iloc[0]
    query = f"""
        SELECT 
            s.stk_pattern,
            s.stk_weight,
            s.stk_labor_cost
        FROM
            konghin.stock s
            JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id
        WHERE 
            p.pur_date >= NOW() - INTERVAL '730 days'
            and s.stk_pattern = '{sample_stk_pattern}'
    """
    a = db.select_raw(query)
display(a)

,stk_id,stk_pattern,stk_weight,map_key,stk_labor_cost,pur_id,pur_date,max_labor_cost,range
0,STK_100229,空心万字手链,6.06,BRACELET | 空心万字手链 | 6,79,PUR_100605,2024-10-01,NaN,1


[2025-12-11 17:18:19] [INFO] Successfully executed query: 
        SELECT 
            s.stk_pattern,
            s.stk_weight,
            s.stk_labor_cost
        FROM
            konghin.stock s
            JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id
        WHERE 
            p.pur_date >= NOW() - INTERVAL '730 days'
            and s.stk_pattern = '空心万字手链'
    


,stk_pattern,stk_weight,stk_labor_cost
0,空心万字手链,6.06,79
1,空心万字手链,5.86,68
2,空心万字手链,5.57,79
